# EchoNext – Predicting left and right ventricular systolic dysfunction from routine ECG measurements

**Outcomes**
- Primary: `lvef_lte_45_flag` (LVEF ≤45%)
- Secondary: `rv_systolic_dysfunction_moderate_or_greater_flag` (moderate or severe RV systolic dysfunction)

**Predictors:** age, sex, ventricular rate, atrial rate, PR interval, QRS duration, QTc

**Data rules (PhysioNet Restricted Health Data Use Agreement 1.5.0)**
- The EchoNext data are **not** included in this repository and must not be redistributed.
- Do not upload the data file to online AI chat services.
- The notebook writes aggregate results only (tables and figures); no individual-level rows are exported.

**How to run:** edit `DATA_PATH` in cell 2, then *Run All*. All tables and figures are saved to `OUT_DIR`.

In [ ]:
# 1) Required packages (run once; see requirements.txt for the versions used in the study)
%pip install -q xgboost shap openpyxl

In [ ]:
# 2) SETTINGS – edit only this cell
import os
DATA_PATH = r"EchoNext_metadata_100k.csv"   # path to the EchoNext metadata file (not included in this repository)
try:                                          # in Google Colab, ask for the CSV with an upload dialog
    from google.colab import files as _colab_files
    if not os.path.exists(DATA_PATH):
        print("Please select the EchoNext_metadata_100k.csv file")
        _up = _colab_files.upload(); DATA_PATH = list(_up.keys())[0]
except ImportError:
    pass
DATASET_VERSION = "1.0.0"                    # dataset version used in the study
OUT_DIR = "echonext_outputs"                 # output folder
SEED = 2026
N_BOOT = 1000                                # bootstrap replicates (1000 in the study)
SHAP_SAMPLE = 5000                           # test records sampled for SHAP

TARGETS = {
    "lvef_lte_45_flag": "LVEF ≤45%",
    "rv_systolic_dysfunction_moderate_or_greater_flag": "RV systolic dysfunction",
}

In [ ]:
# 3) Libraries
import os, sys, platform, warnings
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
warnings.filterwarnings("ignore")

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAS_XGB = False
    print("WARNING: xgboost not installed – HistGradientBoosting will be used instead. xgboost was used in the study.")
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("WARNING: shap not installed – permutation importance will be computed instead. shap was used in the study.")

os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10})
rng = np.random.default_rng(SEED)

def savefig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, name + ".png"), bbox_inches="tight")
    fig.savefig(os.path.join(OUT_DIR, name + ".pdf"), bbox_inches="tight")

FEAT_LABELS = {
    "age_at_ecg": "Age", "male": "Male sex", "ventricular_rate": "Ventricular rate",
    "atrial_rate": "Atrial rate", "pr_interval": "PR interval", "qrs_duration": "QRS duration",
    "qt_corrected": "QTc",
}
DEMO = ["age_at_ecg", "male"]
ECG = ["ventricular_rate", "atrial_rate", "pr_interval", "qrs_duration", "qt_corrected"]
FEATURE_SETS = {"Age + sex": DEMO, "ECG only": ECG, "Age + sex + ECG": DEMO + ECG}
print("xgboost:", HAS_XGB, "| shap:", HAS_SHAP)

## 1. Data loading and analysis set

In [ ]:
raw = pd.read_csv(DATA_PATH)
need = ["ecg_key", "patient_key", "age_at_ecg", "sex", "split"] + ECG + list(TARGETS)
missing_cols = [c for c in need if c not in raw.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"

flow = [("All ECGs in file", len(raw), raw.patient_key.nunique())]
print(raw.split.value_counts())

# 'no_split' = additional ECGs of validation/test patients -> excluded to prevent leakage
d = raw[raw.split.isin(["train", "val", "test"])].copy()
flow.append(("Excluding 'no_split' ECGs", len(d), d.patient_key.nunique()))

# Check that the patient-level splits do not overlap
sets_ = {s: set(d.loc[d.split == s, "patient_key"]) for s in ["train", "val", "test"]}
ov = {f"{a}-{b}": len(sets_[a] & sets_[b]) for a, b in [("train","val"),("train","test"),("val","test")]}
print("Patient overlap (should be 0):", ov)

d["male"] = (d.sex.str.lower() == "male").astype(int)

In [ ]:
# Physiologically implausible values -> set to missing (NaN)
RANGES = {"ventricular_rate": (20, 300), "atrial_rate": (20, 600), "pr_interval": (60, 600),
          "qrs_duration": (40, 260), "qt_corrected": (250, 750)}
clean_rows = []
for c, (lo, hi) in RANGES.items():
    bad = d[c].notna() & ((d[c] < lo) | (d[c] > hi))
    clean_rows.append({"Variable": FEAT_LABELS[c], "Accepted range": f"{lo}–{hi}",
                       "Set to missing (n)": int(bad.sum()),
                       "Missing after cleaning (%)": None})
    d.loc[bad, c] = np.nan
for r, c in zip(clean_rows, RANGES):
    r["Missing after cleaning (%)"] = round(100 * d[c].isna().mean(), 2)
clean_tab = pd.DataFrame(clean_rows)

# ECGs with a missing outcome label are excluded
before = len(d)
d = d.dropna(subset=list(TARGETS))
flow.append(("Excluding ECGs with missing target labels", len(d), d.patient_key.nunique()))
for s in ["train", "val", "test"]:
    flow.append((f"  {s}", int((d.split == s).sum()), d.loc[d.split == s, "patient_key"].nunique()))
flow_tab = pd.DataFrame(flow, columns=["Step", "ECGs", "Patients"])
display(flow_tab); display(clean_tab)

ecg_per_pat = d[d.split == "train"].groupby("patient_key").size()
print("Train: ECGs per patient – median [IQR]:",
      ecg_per_pat.median(), ecg_per_pat.quantile([.25, .75]).tolist(), "max:", ecg_per_pat.max())

tr, va, te = (d[d.split == s].copy() for s in ["train", "val", "test"])

## 2. Table 1

In [ ]:
CONT = ["age_at_ecg"] + ECG
def fmt_cont(x):
    x = x.dropna()
    return f"{x.median():.0f} [{x.quantile(.25):.0f}–{x.quantile(.75):.0f}]"
def fmt_bin(x):
    return f"{int(x.sum()):,} ({100 * x.mean():.1f})"

def table1(groups):
    rows = {}
    rows["ECGs, n"] = {g: f"{len(x):,}" for g, x in groups.items()}
    rows["Patients, n"] = {g: f"{x.patient_key.nunique():,}" for g, x in groups.items()}
    rows["Male sex, n (%)"] = {g: fmt_bin(x.male) for g, x in groups.items()}
    for c in CONT:
        rows[f"{FEAT_LABELS[c]}, median [IQR]"] = {g: fmt_cont(x[c]) for g, x in groups.items()}
    for t, lab in TARGETS.items():
        rows[f"{lab}, n (%)"] = {g: fmt_bin(x[t]) for g, x in groups.items()}
    return pd.DataFrame(rows).T

# 1a: by data split
t1a = table1({"Overall": d, "Train": tr, "Validation": va, "Test": te})
display(t1a)

# 1b: test set by outcome status (one ECG per patient -> independent observations, so p values are valid)
def table1_by_target(df, t):
    g1, g0 = df[df[t] == 1], df[df[t] == 0]
    tab = table1({f"{TARGETS[t]}: yes": g1, f"{TARGETS[t]}: no": g0}).drop(index=[f"{l}, n (%)" for l in TARGETS.values()])
    p = {"Male sex, n (%)": stats.chi2_contingency(pd.crosstab(df.male, df[t]))[1]}
    for c in CONT:
        p[f"{FEAT_LABELS[c]}, median [IQR]"] = stats.mannwhitneyu(g1[c].dropna(), g0[c].dropna())[1]
    tab["p"] = pd.Series(p).map(lambda v: "<0.001" if v < 0.001 else f"{v:.3f}")
    return tab.fillna("")

t1b = {t: table1_by_target(te, t) for t in TARGETS}
for t in TARGETS: display(t1b[t])

## 3. Descriptive figure: prevalence by QRS duration and age group (Figure 1)

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; den = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / den; h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return c - h, c + h

d["qrs_cat"] = pd.cut(d.qrs_duration, [0, 100, 120, 150, 1000], labels=["≤100", "101–120", "121–150", ">150"])
d["age_cat"] = pd.cut(d.age_at_ecg, [0, 49, 69, 200], labels=["<50", "50–69", "≥70"])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
prev_rows = []
for ax, (cat, xlab) in zip(axes, [("qrs_cat", "QRS duration (ms)"), ("age_cat", "Age (years)")]):
    levels = d[cat].cat.categories
    w = 0.38
    for j, (t, lab) in enumerate(TARGETS.items()):
        ps, lo, hi = [], [], []
        for lv in levels:
            x = d.loc[d[cat] == lv, t]
            k, n = int(x.sum()), len(x); a, b = wilson(k, n)
            ps.append(100 * k / n); lo.append(100 * a); hi.append(100 * b)
            prev_rows.append({"Grouping": xlab, "Level": lv, "Outcome": lab, "n": n, "Events": k,
                              "Prevalence %": round(100 * k / n, 1), "95% CI": f"{100*a:.1f}–{100*b:.1f}"})
        xs = np.arange(len(levels)) + (j - 0.5) * w
        ax.bar(xs, ps, w, label=lab, yerr=[np.array(ps) - lo, np.array(hi) - ps], capsize=3)
    ax.set_xticks(range(len(levels))); ax.set_xticklabels(levels); ax.set_xlabel(xlab)
axes[0].set_ylabel("Prevalence (%)"); axes[0].legend(frameon=False)
fig.tight_layout(); savefig(fig, "fig1_prevalence"); plt.show()
prev_tab = pd.DataFrame(prev_rows)

## 4. Models

For each outcome: 3 predictor sets × 2 models (logistic regression, XGBoost). Training = train split; early stopping and threshold selection = validation split; evaluation = test split.

In [ ]:
def fit_lr(X, y):
    m = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                      LogisticRegression(max_iter=2000))
    return m.fit(X, y)

def fit_gbm(Xtr, ytr, Xva, yva):
    if HAS_XGB:
        m = xgb.XGBClassifier(n_estimators=3000, learning_rate=0.03, max_depth=4, min_child_weight=5,
                              subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                              tree_method="hist", eval_metric="auc", early_stopping_rounds=100,
                              random_state=SEED, n_jobs=-1)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    else:
        m = HistGradientBoostingClassifier(max_iter=1000, learning_rate=0.03, max_depth=4,
                                           early_stopping=True, validation_fraction=0.1,
                                           n_iter_no_change=50, random_state=SEED)
        m.fit(Xtr, ytr)
    return m

def boot_idx(n, B, seed=SEED):
    r = np.random.default_rng(seed)
    return [r.integers(0, n, n) for _ in range(B)]

def metric_ci(y, p, fn, idx):
    est = fn(y, p); vals = []
    for i in idx:
        if y[i].min() == y[i].max(): continue
        vals.append(fn(y[i], p[i]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return est, lo, hi, np.array(vals)

def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return thr[np.argmax(tpr - fpr)]

def threshold_metrics(y, p, thr):
    pred = p >= thr
    tp = int(((pred == 1) & (y == 1)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    f = lambda k, n: f"{100*k/n:.1f} ({100*wilson(k,n)[0]:.1f}–{100*wilson(k,n)[1]:.1f})" if n else ""
    return {"Sensitivity %": f(tp, tp + fn), "Specificity %": f(tn, tn + fp),
            "PPV %": f(tp, tp + fp), "NPV %": f(tn, tn + fn)}

In [ ]:
models, preds, results = {}, {}, []
IDX = boot_idx(len(te), N_BOOT)
fmt = lambda e, l, h, k=3: f"{e:.{k}f} ({l:.{k}f}–{h:.{k}f})"

for t, tlab in TARGETS.items():
    ytr, yva, yte = tr[t].values, va[t].values, te[t].values
    for fs_name, cols in FEATURE_SETS.items():
        for mname in ["Logistic regression", "XGBoost" if HAS_XGB else "Gradient boosting"]:
            if mname == "Logistic regression":
                m = fit_lr(tr[cols], ytr)
            else:
                m = fit_gbm(tr[cols], ytr, va[cols], yva)
            pva = m.predict_proba(va[cols])[:, 1]; pte = m.predict_proba(te[cols])[:, 1]
            key = (t, fs_name, mname); models[key] = m; preds[key] = pte
            auc = metric_ci(yte, pte, roc_auc_score, IDX)
            ap = metric_ci(yte, pte, average_precision_score, IDX)
            br = metric_ci(yte, pte, brier_score_loss, IDX)
            thr = youden_threshold(yva, pva)
            row = {"Outcome": tlab, "Features": fs_name, "Model": mname,
                   "Test n": len(yte), "Events": int(yte.sum()), "Prevalence %": round(100 * yte.mean(), 1),
                   "AUROC (95% CI)": fmt(*auc[:3]), "AUPRC (95% CI)": fmt(*ap[:3]), "Brier (95% CI)": fmt(*br[:3]),
                   "Threshold (val Youden)": round(float(thr), 4)}
            row.update(threshold_metrics(yte, pte, thr))
            results.append(row)
            print(f"{tlab:28s} | {fs_name:16s} | {mname:20s} | AUROC {fmt(*auc[:3])}")

res_tab = pd.DataFrame(results)
display(res_tab)

In [ ]:
# Paired bootstrap: full model (Age+sex+ECG) vs simpler models, and LR vs XGBoost
GBM = "XGBoost" if HAS_XGB else "Gradient boosting"
def paired_diff(y, p1, p2, idx):
    est = roc_auc_score(y, p1) - roc_auc_score(y, p2); vals = []
    for i in idx:
        if y[i].min() == y[i].max(): continue
        vals.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
    vals = np.array(vals); lo, hi = np.percentile(vals, [2.5, 97.5])
    p = 2 * min((vals <= 0).mean(), (vals >= 0).mean()); p = max(p, 1 / len(vals))
    return est, lo, hi, p

comp = []
for t, tlab in TARGETS.items():
    y = te[t].values
    pairs = [((t, "Age + sex + ECG", GBM), (t, "Age + sex", GBM)),
             ((t, "Age + sex + ECG", GBM), (t, "ECG only", GBM)),
             ((t, "Age + sex + ECG", GBM), (t, "Age + sex + ECG", "Logistic regression"))]
    for a, b in pairs:
        e, lo, hi, p = paired_diff(y, preds[a], preds[b], IDX)
        comp.append({"Outcome": tlab, "Comparison": f"{a[1]} [{a[2]}] − {b[1]} [{b[2]}]",
                     "ΔAUROC (95% CI)": fmt(e, lo, hi), "Bootstrap p": f"<{1/N_BOOT:.3f}" if p <= 1/N_BOOT else f"{p:.3f}"})
comp_tab = pd.DataFrame(comp); display(comp_tab)

## 5. ROC curves and calibration (Figures 2 and 3A)

In [ ]:
fig, axes = plt.subplots(1, len(TARGETS), figsize=(5 * len(TARGETS), 4.5))
axes = np.atleast_1d(axes)
for ax, (t, tlab) in zip(axes, TARGETS.items()):
    y = te[t].values
    for fs_name in FEATURE_SETS:
        p = preds[(t, fs_name, GBM)]; fpr, tpr, _ = roc_curve(y, p)
        ax.plot(fpr, tpr, label=f"{fs_name} (AUROC {roc_auc_score(y, p):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.7)
    ax.set_title(tlab); ax.set_xlabel("1 − Specificity"); ax.set_ylabel("Sensitivity"); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); savefig(fig, "fig2_roc"); plt.show()

fig, axes = plt.subplots(1, len(TARGETS), figsize=(5 * len(TARGETS), 4.5))
axes = np.atleast_1d(axes)
for ax, (t, tlab) in zip(axes, TARGETS.items()):
    y = te[t].values
    for mname in ["Logistic regression", GBM]:
        p = preds[(t, "Age + sex + ECG", mname)]
        obs, pr = calibration_curve(y, p, n_bins=10, strategy="quantile")
        ax.plot(pr, obs, "o-", ms=4, label=mname)
    lim = max(ax.get_xlim()[1], ax.get_ylim()[1]); ax.plot([0, lim], [0, lim], "k--", lw=0.7)
    ax.set_title(tlab); ax.set_xlabel("Predicted probability"); ax.set_ylabel("Observed proportion"); ax.legend(frameon=False)
fig.tight_layout(); savefig(fig, "fig3_calibration"); plt.show()

## 6. Explainability (SHAP; Figures 4 and 5)

In [ ]:
FULL = DEMO + ECG
imp_rows = []
for t, tlab in TARGETS.items():
    m = models[(t, "Age + sex + ECG", GBM)]
    Xs = te[FULL].sample(min(SHAP_SAMPLE, len(te)), random_state=SEED)
    Xs_lab = Xs.rename(columns=FEAT_LABELS)
    if HAS_SHAP:
        try:
            sv = shap.TreeExplainer(m).shap_values(Xs)
            if isinstance(sv, list): sv = sv[1]
        except Exception as e:   # if shap and xgboost versions are incompatible, xgboost's own SHAP computation is used
            print("TreeExplainer error, using xgboost pred_contribs:", e)
            sv = m.get_booster().predict(xgb.DMatrix(Xs), pred_contribs=True)[:, :-1]
        mean_abs = pd.Series(np.abs(sv).mean(0), index=Xs_lab.columns).sort_values(ascending=False)
        plt.figure(); shap.summary_plot(sv, Xs_lab, show=False, plot_size=(7, 4.5))
        plt.title(tlab); savefig(plt.gcf(), f"fig4_shap_beeswarm_{t}"); plt.show()
        top = mean_abs.index[:4]
        fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
        for ax, f in zip(axes, top):
            j = list(Xs_lab.columns).index(f)
            ax.scatter(Xs_lab[f], sv[:, j], s=3, alpha=0.3)
            ax.axhline(0, color="k", lw=0.6); ax.set_xlabel(f); ax.set_ylabel("SHAP value (log-odds)")
        fig.suptitle(tlab); fig.tight_layout(); savefig(fig, f"fig5_shap_dependence_{t}"); plt.show()
        method = "mean |SHAP|"
    else:
        from sklearn.inspection import permutation_importance
        pi = permutation_importance(m, Xs, te.loc[Xs.index, t], scoring="roc_auc", n_repeats=10, random_state=SEED)
        mean_abs = pd.Series(pi.importances_mean, index=Xs_lab.columns).sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(6, 3.5)); mean_abs[::-1].plot.barh(ax=ax)
        ax.set_xlabel("Permutation importance (ΔAUROC)"); ax.set_title(tlab)
        fig.tight_layout(); savefig(fig, f"fig4_importance_{t}"); plt.show()
        method = "permutation ΔAUROC"
    for rank, (f, v) in enumerate(mean_abs.items(), 1):
        imp_rows.append({"Outcome": tlab, "Rank": rank, "Feature": f, method: round(float(v), 4)})
imp_tab = pd.DataFrame(imp_rows); display(imp_tab)

## 7. Subgroup analyses (test set, full XGBoost model; Figure 6)

In [ ]:
te["qrs_cat"] = pd.cut(te.qrs_duration, [0, 119.5, 1000], labels=["<120 ms", "≥120 ms"])
te["age_cat"] = pd.cut(te.age_at_ecg, [0, 49, 69, 200], labels=["<50", "50–69", "≥70"])
te["sex_cat"] = np.where(te.male == 1, "Male", "Female")
SUB = {"Sex": "sex_cat", "Age group": "age_cat", "QRS": "qrs_cat"}

sub_rows = []
for t, tlab in TARGETS.items():
    p_all = pd.Series(preds[(t, "Age + sex + ECG", GBM)], index=te.index)
    for gname, col in SUB.items():
        for lv in (te[col].cat.categories if hasattr(te[col], "cat") else sorted(te[col].dropna().unique())):
            ix = te.index[te[col] == lv]; y = te.loc[ix, t].values; p = p_all.loc[ix].values
            if y.sum() < 10 or (1 - y).sum() < 10:
                sub_rows.append({"Outcome": tlab, "Subgroup": gname, "Level": lv, "n": len(y), "Events": int(y.sum()),
                                 "AUROC": np.nan, "lo": np.nan, "hi": np.nan}); continue
            e, lo, hi, _ = metric_ci(y, p, roc_auc_score, boot_idx(len(y), N_BOOT))
            sub_rows.append({"Outcome": tlab, "Subgroup": gname, "Level": lv, "n": len(y), "Events": int(y.sum()),
                             "AUROC": e, "lo": lo, "hi": hi})
sub_tab = pd.DataFrame(sub_rows)
sub_tab["AUROC (95% CI)"] = [fmt(a, b, c) if pd.notna(a) else "too few events"
                             for a, b, c in zip(sub_tab.AUROC, sub_tab.lo, sub_tab.hi)]

fig, axes = plt.subplots(1, len(TARGETS), figsize=(5.5 * len(TARGETS), 3.8), sharey=True)
axes = np.atleast_1d(axes)
for ax, (t, tlab) in zip(axes, TARGETS.items()):
    s = sub_tab[(sub_tab.Outcome == tlab) & sub_tab.AUROC.notna()].reset_index(drop=True)
    ys = np.arange(len(s))[::-1]
    ax.errorbar(s.AUROC, ys, xerr=[s.AUROC - s.lo, s.hi - s.AUROC], fmt="o", capsize=3)
    ax.set_yticks(ys); ax.set_yticklabels(s.Subgroup + ": " + s.Level.astype(str))
    ax.axvline(roc_auc_score(te[t], preds[(t, "Age + sex + ECG", GBM)]), color="grey", ls="--", lw=0.8)
    ax.set_xlabel("AUROC (95% CI)"); ax.set_title(tlab)
fig.tight_layout(); savefig(fig, "fig6_subgroups"); plt.show()
display(sub_tab.drop(columns=["AUROC", "lo", "hi"]))

## 8. Sensitivity analysis: one ECG per patient in the training set

Some patients have many ECGs in the training set. The model is retrained with one randomly selected ECG per patient.

In [ ]:
tr1 = tr.groupby("patient_key", group_keys=False).sample(n=1, random_state=SEED)
sens_rows = []
for t, tlab in TARGETS.items():
    m = fit_gbm(tr1[FULL], tr1[t].values, va[FULL], va[t].values)
    p = m.predict_proba(te[FULL])[:, 1]
    e, lo, hi, _ = metric_ci(te[t].values, p, roc_auc_score, IDX)
    main = [r for r in results if r["Outcome"] == tlab and r["Features"] == "Age + sex + ECG" and r["Model"] == GBM][0]
    sens_rows.append({"Outcome": tlab, "Main analysis AUROC": main["AUROC (95% CI)"],
                      "One ECG per patient AUROC": fmt(e, lo, hi), "Training ECGs (one per patient)": len(tr1)})
sens_tab = pd.DataFrame(sens_rows); display(sens_tab)

## 9. Save outputs and software versions

In [ ]:
xlsx = os.path.join(OUT_DIR, "echonext_results.xlsx")
with pd.ExcelWriter(xlsx) as w:
    flow_tab.to_excel(w, sheet_name="Flow", index=False)
    clean_tab.to_excel(w, sheet_name="Cleaning", index=False)
    t1a.to_excel(w, sheet_name="Table1_splits")
    for t in TARGETS: t1b[t].to_excel(w, sheet_name=("Table1_" + t)[:31])
    prev_tab.to_excel(w, sheet_name="Prevalence", index=False)
    res_tab.to_excel(w, sheet_name="Performance", index=False)
    comp_tab.to_excel(w, sheet_name="AUROC_comparisons", index=False)
    imp_tab.to_excel(w, sheet_name="Importance", index=False)
    sub_tab.drop(columns=["AUROC", "lo", "hi"]).to_excel(w, sheet_name="Subgroups", index=False)
    sens_tab.to_excel(w, sheet_name="Sensitivity", index=False)

info = {"EchoNext version": DATASET_VERSION, "Python": platform.python_version(), "pandas": pd.__version__,
        "numpy": np.__version__, "scikit-learn": sklearn.__version__, "matplotlib": matplotlib.__version__,
        "xgboost": xgb.__version__ if HAS_XGB else "not installed", "shap": shap.__version__ if HAS_SHAP else "not installed",
        "Seed": SEED, "Bootstrap replicates": N_BOOT}
pd.Series(info).to_csv(os.path.join(OUT_DIR, "software_versions.csv"), header=False)
print(pd.Series(info).to_string())
print("\nSaved to:", os.path.abspath(OUT_DIR))
print(sorted(os.listdir(OUT_DIR)))

Outputs in `echonext_outputs` (`echonext_results.xlsx` and figures) are aggregate results only. Do not share the raw data file.

## 10. Additional analyses (calibration intercept/slope, high-sensitivity thresholds, missing vs complete records, reference model, prevalence)

In [ ]:
# ===== Additional analyses requested during review =====
from scipy.special import logit as _lg, expit as _ex
from scipy.optimize import minimize_scalar

def _recal(t):
    m = models[(t, "Age + sex + ECG", GBM)]
    pva = np.clip(m.predict_proba(va[FULL])[:, 1], 1e-6, 1 - 1e-6)
    pte = np.clip(preds[(t, "Age + sex + ECG", GBM)], 1e-6, 1 - 1e-6)
    cal = LogisticRegression(C=1e6).fit(_lg(pva).reshape(-1, 1), va[t])
    return pva, pte, cal

def cal_int_slope(y, p, idx):
    lp = _lg(np.clip(p, 1e-6, 1 - 1e-6))
    def one(yy, ll):
        slope = LogisticRegression(C=1e6).fit(ll.reshape(-1, 1), yy).coef_[0, 0]
        nll = lambda a: -np.sum(yy * (a + ll) - np.log1p(np.exp(a + ll)))
        inter = minimize_scalar(nll, bounds=(-5, 5), method="bounded").x
        return inter, slope
    i0, s0 = one(y, lp); bi, bs = [], []
    for i in idx[:500]:
        a, b = one(y[i], lp[i]); bi.append(a); bs.append(b)
    f = lambda e, v: f"{e:.3f} ({np.percentile(v, 2.5):.3f} to {np.percentile(v, 97.5):.3f})"
    return f(i0, bi), f(s0, bs)

# 1) Calibration intercept and slope
rows = []
for t, tlab in TARGETS.items():
    y = te[t].values.astype(int); pva, pte, cal = _recal(t)
    pc = cal.predict_proba(_lg(pte).reshape(-1, 1))[:, 1]
    for lab, p in [("Original", pte), ("Recalibrated", pc)]:
        ci, cs = cal_int_slope(y, p, IDX)
        rows.append({"Outcome": tlab, "Version": lab, "Calibration intercept (95% CI)": ci, "Calibration slope (95% CI)": cs})
cal_tab = pd.DataFrame(rows); display(cal_tab)

# 2) Thresholds targeting high sensitivity (selected on the validation set, applied to the test set)
rows = []
for t, tlab in TARGETS.items():
    yv, y = va[t].values.astype(int), te[t].values.astype(int)
    pva = models[(t, "Age + sex + ECG", GBM)].predict_proba(va[FULL])[:, 1]
    pte = preds[(t, "Age + sex + ECG", GBM)]
    for target_sens in [0.90, 0.95]:
        thr = np.quantile(pva[yv == 1], 1 - target_sens)
        pred = pte >= thr
        tp = int((pred & (y == 1)).sum()); fn = int((~pred & (y == 1)).sum())
        tn = int((~pred & (y == 0)).sum()); fp = int((pred & (y == 0)).sum())
        cs = lambda k, n: f"{100*k/n:.1f} ({100*wilson(k,n)[0]:.1f}–{100*wilson(k,n)[1]:.1f})"
        rows.append({"Outcome": tlab, "Target sensitivity (validation)": f"{int(target_sens*100)}%",
                     "Test sensitivity %": cs(tp, tp + fn), "Test specificity %": cs(tn, tn + fp),
                     "PPV %": cs(tp, tp + fp), "NPV %": cs(tn, tn + fn),
                     "Flagged for echo %": f"{100*pred.mean():.1f}", "Missed cases (n)": fn})
hs_tab = pd.DataFrame(rows); display(hs_tab)

# 3) Missing data: records with missing vs complete ECG measurements in the test set
te_inc = te[ECG + DEMO].isna().any(axis=1)
rows = []
for t, tlab in TARGETS.items():
    y = te[t].values.astype(int); p = preds[(t, "Age + sex + ECG", GBM)]
    for lab, mask in [("Complete records", ~te_inc.values), ("Records with ≥1 missing ECG value", te_inc.values)]:
        yy, pp = y[mask], p[mask]
        k, n = int(yy.sum()), len(yy)
        auc = metric_ci(yy, pp, roc_auc_score, boot_idx(n, 500))[:3] if 10 <= k <= n - 10 else (np.nan,) * 3
        rows.append({"Outcome": tlab, "Subset": lab, "n": n, "Prevalence %": f"{100*k/n:.1f} ({100*wilson(k,n)[0]:.1f}–{100*wilson(k,n)[1]:.1f})",
                     "AUROC (95% CI)": fmt(*auc) if not np.isnan(auc[0]) else "too few events"})
miss_tab = pd.DataFrame(rows); display(miss_tab)

# 4) Reference model with the predictors of Kim et al. on the SAME test set: LR (age, sex, QRS, PR, QTc), trained on complete records only
KIM = ["age_at_ecg", "male", "qrs_duration", "pr_interval", "qt_corrected"]
rows = []
for t, tlab in TARGETS.items():
    trc = tr.dropna(subset=KIM)
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(trc[KIM], trc[t])
    y = te[t].values.astype(int); p_main = preds[(t, "Age + sex + ECG", GBM)]
    comp = te[KIM].notna().all(axis=1).values
    # (a) complete-record test subset – both models in the same patients
    yy = y[comp]; pk = lr.predict_proba(te.loc[comp, KIM])[:, 1]; pm = p_main[comp]
    idx = boot_idx(len(yy), N_BOOT)
    ek = metric_ci(yy, pk, roc_auc_score, idx)[:3]; em = metric_ci(yy, pm, roc_auc_score, idx)[:3]
    dif = np.array([roc_auc_score(yy[i], pm[i]) - roc_auc_score(yy[i], pk[i]) for i in idx if 0 < yy[i].sum() < len(i)])
    # (b) full test set – missing values in the reference model imputed with training medians
    Xf = te[KIM].fillna(trc[KIM].median()); pk_all = lr.predict_proba(Xf)[:, 1]
    ea = metric_ci(y, pk_all, roc_auc_score, IDX)[:3]
    rows.append({"Outcome": tlab, "Training ECGs (complete cases)": len(trc),
                 "Test subset": f"complete cases (n={comp.sum()})",
                 "Kim-like LR AUROC": fmt(*ek), "Our model AUROC": fmt(*em),
                 "ΔAUROC (95% CI)": f"{em[0]-ek[0]:.3f} ({np.percentile(dif,2.5):.3f}–{np.percentile(dif,97.5):.3f})",
                 "Kim-like LR AUROC, full test set (median-imputed)": fmt(*ea)})
kim_tab = pd.DataFrame(rows); display(kim_tab)

# 5) Prevalence: training set with one ECG per patient
tr1_ = tr.groupby("patient_key", group_keys=False).sample(n=1, random_state=SEED)
prev_rows = [{"Outcome": tlab, "Train (all ECGs) %": round(100*tr[t].mean(), 1),
              "Train (one ECG per patient) %": round(100*tr1_[t].mean(), 1),
              "Validation %": round(100*va[t].mean(), 1), "Test %": round(100*te[t].mean(), 1)} for t, tlab in TARGETS.items()]
prev2_tab = pd.DataFrame(prev_rows); display(prev2_tab)

with pd.ExcelWriter(os.path.join(OUT_DIR, "reviewer_analyses.xlsx")) as w:
    cal_tab.to_excel(w, sheet_name="Calibration_int_slope", index=False)
    hs_tab.to_excel(w, sheet_name="High_sensitivity", index=False)
    miss_tab.to_excel(w, sheet_name="Missing_vs_complete", index=False)
    kim_tab.to_excel(w, sheet_name="Kim_like_comparison", index=False)
    prev2_tab.to_excel(w, sheet_name="Prevalence_by_set", index=False)
print("Saved:", os.path.join(OUT_DIR, "reviewer_analyses.xlsx"))

In [ ]:
# Download the result file (Google Colab only)
try:
    from google.colab import files as _f
    _f.download(os.path.join(OUT_DIR, "reviewer_analyses.xlsx"))
except ImportError:
    print("Not in Colab: the file is in", OUT_DIR)

## 11. Comparison with simple ECG rules – threshold selected on the validation set (Table 4)

In [ ]:
# ===== Comparison with simple ECG rules: threshold selected on the VALIDATION set, evaluated on the TEST set =====
def _rules(df):
    q, c = df.qrs_duration.fillna(0), df.qt_corrected.fillna(0)
    return {"QRS >100 ms": (q > 100).values, "QRS ≥120 ms": (q >= 120).values,
            "QTc ≥470 ms": (c >= 470).values, "QRS ≥120 ms or QTc ≥470 ms": ((q >= 120) | (c >= 470)).values}
def _cs(k, n):
    lo, hi = wilson(k, n); return f"{100*k/n:.1f} ({100*lo:.1f}–{100*hi:.1f})"
def _conf(y, pred):
    pred = np.asarray(pred, bool)
    return (int((pred & (y == 1)).sum()), int((~pred & (y == 1)).sum()),
            int((~pred & (y == 0)).sum()), int((pred & (y == 0)).sum()))
R_va, R_te = _rules(va), _rules(te)
rows = []
for t, tlab in TARGETS.items():
    m = models[(t, "Age + sex + ECG", GBM)]
    yv, y = va[t].values.astype(int), te[t].values.astype(int)
    pv, pt = m.predict_proba(va[FULL])[:, 1], preds[(t, "Age + sex + ECG", GBM)]
    for r in R_va:
        tp, fn, tn, fp = _conf(yv, R_va[r]); sens_val = tp / (tp + fn)
        thr = np.quantile(pv[yv == 1], 1 - sens_val)          # threshold matching the rule's validation sensitivity
        rtp, rfn, rtn, rfp = _conf(y, R_te[r]); mtp, mfn, mtn, mfp = _conf(y, pt >= thr)
        # paired bootstrap: specificity difference (model − rule) in the test set
        diffs, dsens = [], []
        for i in IDX:
            yy = y[i]; neg, pos = yy == 0, yy == 1
            if neg.sum() == 0 or pos.sum() == 0: continue
            diffs.append(((pt[i] < thr) & neg).sum() / neg.sum() - ((~R_te[r][i]) & neg).sum() / neg.sum())
            dsens.append(((pt[i] >= thr) & pos).sum() / pos.sum() - (R_te[r][i] & pos).sum() / pos.sum())
        d0 = mtn / (mtn + mfp) - rtn / (rtn + rfp); s0 = mtp / (mtp + mfn) - rtp / (rtp + rfn)
        rows.append({"Outcome": tlab, "Rule": r,
                     "Rule sensitivity % (test)": _cs(rtp, rtp + rfn), "Rule specificity % (test)": _cs(rtn, rtn + rfp),
                     "Model sensitivity % (test)": _cs(mtp, mtp + mfn), "Model specificity % (test)": _cs(mtn, mtn + mfp),
                     "Δ sensitivity, model − rule (95% CI)": f"{100*s0:.1f} ({100*np.percentile(dsens,2.5):.1f} to {100*np.percentile(dsens,97.5):.1f})",
                     "Δ specificity, model − rule (95% CI)": f"{100*d0:.1f} ({100*np.percentile(diffs,2.5):.1f} to {100*np.percentile(diffs,97.5):.1f})"})
rules_val_tab = pd.DataFrame(rows); display(rules_val_tab)
rules_val_tab.to_excel(os.path.join(OUT_DIR, "rules_validation_threshold.xlsx"), index=False)
try:
    from google.colab import files as _f
    _f.download(os.path.join(OUT_DIR, "rules_validation_threshold.xlsx"))
except ImportError:
    print("Not in Colab: the file is in", OUT_DIR)


## 12. Recalibration on the validation set (Figure 3B)

Platt scaling of the full XGBoost model on the validation set; mean predicted risk, Brier score and AUROC before and after recalibration.

In [ ]:
from scipy.special import logit
fig, axes = plt.subplots(1, len(TARGETS), figsize=(10, 4.5))
rows = []
for ax, (t, tlab) in zip(axes, TARGETS.items()):
    m = models[(t, "Age + sex + ECG", GBM)]
    pva = np.clip(m.predict_proba(va[FULL])[:, 1], 1e-6, 1 - 1e-6)
    pte = np.clip(preds[(t, "Age + sex + ECG", GBM)], 1e-6, 1 - 1e-6)
    cal = LogisticRegression(C=1e6).fit(logit(pva).reshape(-1, 1), va[t])
    pte_c = cal.predict_proba(logit(pte).reshape(-1, 1))[:, 1]
    y = te[t].values
    for p, lab in [(pte, "Original"), (pte_c, "Recalibrated (validation set)")]:
        o, pr = calibration_curve(y, p, n_bins=10, strategy="quantile")
        ax.plot(pr, o, "o-", ms=4, label=lab)
        rows.append({"Outcome": tlab, "Version": lab, "Mean predicted %": round(100 * p.mean(), 1),
                     "Observed %": round(100 * y.mean(), 1), "Brier": round(brier_score_loss(y, p), 4),
                     "AUROC": round(roc_auc_score(y, p), 3)})
    lim = max(ax.get_xlim()[1], ax.get_ylim()[1]); ax.plot([0, lim], [0, lim], "k--", lw=0.7)
    ax.set_title(tlab); ax.set_xlabel("Predicted probability"); ax.set_ylabel("Observed proportion"); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); savefig(fig, "fig3b_recalibration"); plt.show()
recal_tab = pd.DataFrame(rows); display(recal_tab)
recal_tab.to_excel(os.path.join(OUT_DIR, "recalibration.xlsx"), index=False)

## 13. Decision curve analysis (Figure 7)

**Note – not all of this cell is reported.**
- *First part (NOT reported):* compares the model with simple rules at the same sensitivity read from the **test-set** ROC curve (`rules_and_dca.xlsx`, sheet `Rules_vs_model`). This earlier version was superseded by Section 11 (threshold selected on the **validation** set), which is the comparison reported in Table 4. It is kept unchanged for transparency.
- *Second part (reported):* decision curve analysis with recalibrated probabilities (Figure 7; sheets `DCA_selected` and `DCA_full`).

In [ ]:
# ===== Simple clinical rules + decision curve analysis (DCA) =====
# NOTE: The rule comparison in the first part of this cell (rule_tab, sheet 'Rules_vs_model') reads the
# matched sensitivity from the TEST-set ROC curve. It is an earlier version that is NOT reported in the
# manuscript; Table 4 comes from Section 11 (threshold selected on the VALIDATION set).
# The decision curve analysis in the second part of this cell IS reported (Figure 7).
from scipy.special import logit

def ci_str(k, n):
    lo, hi = wilson(k, n)
    return f"{100*k/n:.1f} ({100*lo:.1f}–{100*hi:.1f})" if n else ""

def conf(y, pred):
    pred = np.asarray(pred, bool); y = np.asarray(y, int)
    tp = int((pred & (y == 1)).sum()); fn = int((~pred & (y == 1)).sum())
    tn = int((~pred & (y == 0)).sum()); fp = int((pred & (y == 0)).sum())
    return tp, fn, tn, fp

def recalibrated(t):
    m = models[(t, "Age + sex + ECG", GBM)]
    pva = np.clip(m.predict_proba(va[FULL])[:, 1], 1e-6, 1 - 1e-6)
    pte = np.clip(preds[(t, "Age + sex + ECG", GBM)], 1e-6, 1 - 1e-6)
    cal = LogisticRegression(C=1e6).fit(logit(pva).reshape(-1, 1), va[t])
    return cal.predict_proba(logit(pte).reshape(-1, 1))[:, 1]

# Simple rules (missing value = rule negative)
qrs, qtc = te.qrs_duration.fillna(0), te.qt_corrected.fillna(0)
RULES = {
    "QRS >100 ms": qrs > 100,
    "QRS ≥120 ms": qrs >= 120,
    "QTc ≥470 ms": qtc >= 470,
    "QRS ≥120 ms or QTc ≥470 ms": (qrs >= 120) | (qtc >= 470),
}

rule_rows = []
for t, tlab in TARGETS.items():
    y = te[t].values.astype(int)
    p = preds[(t, "Age + sex + ECG", GBM)]
    fpr, tpr, thr = roc_curve(y, p)
    for rname, r in RULES.items():
        tp, fn, tn, fp = conf(y, r.values)
        sens = tp / (tp + fn)
        # Model specificity at the SAME sensitivity as the rule (from the test-set ROC curve)
        j = np.argmax(tpr >= sens); mtp, mfn, mtn, mfp = conf(y, p >= thr[j])
        rule_rows.append({"Outcome": tlab, "Rule": rname,
                          "Rule sensitivity %": ci_str(tp, tp + fn), "Rule specificity %": ci_str(tn, tn + fp),
                          "Rule PPV %": ci_str(tp, tp + fp), "Rule NPV %": ci_str(tn, tn + fn),
                          "Model sensitivity % (matched)": ci_str(mtp, mtp + mfn),
                          "Model specificity % at same sensitivity": ci_str(mtn, mtn + mfp),
                          "Model NPV % at same sensitivity": ci_str(mtn, mtn + mfn)})
rule_tab = pd.DataFrame(rule_rows); display(rule_tab)

# Decision curve analysis (recalibrated probabilities)
PT = np.round(np.arange(0.01, 0.51, 0.01), 2)
def net_benefit(y, pred, pt):
    tp, fn, tn, fp = conf(y, pred); n = len(y)
    return tp / n - fp / n * pt / (1 - pt)

dca_rows = []
fig, axes = plt.subplots(1, len(TARGETS), figsize=(11, 4.3))
axes = np.atleast_1d(axes)
for ax, (t, tlab) in zip(axes, TARGETS.items()):
    y = te[t].values.astype(int); pc = recalibrated(t); prev = y.mean()
    curves = {"Model (age + sex + ECG)": [net_benefit(y, pc >= pt, pt) for pt in PT],
              "Treat all (echo for everyone)": [prev - (1 - prev) * pt / (1 - pt) for pt in PT],
              "QRS ≥120 ms or QTc ≥470 ms": [net_benefit(y, RULES["QRS ≥120 ms or QTc ≥470 ms"].values, pt) for pt in PT]}
    for name, nb in curves.items():
        ax.plot(PT, nb, label=name, lw=1.8 if name.startswith("Model") else 1.2)
        for pt, v in zip(PT, nb):
            dca_rows.append({"Outcome": tlab, "Strategy": name, "Threshold probability": pt, "Net benefit": round(v, 4)})
    ax.axhline(0, color="k", lw=0.8, label="Treat none")
    ax.set_ylim(-0.02, prev * 1.1); ax.set_xlim(0, 0.5)
    ax.set_xlabel("Threshold probability"); ax.set_ylabel("Net benefit"); ax.set_title(tlab)
    ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); savefig(fig, "fig7_decision_curve"); plt.show()
dca_tab = pd.DataFrame(dca_rows)

# Summary: net benefit at selected thresholds
dca_sel = dca_tab[dca_tab["Threshold probability"].isin([0.05, 0.10, 0.15, 0.20, 0.30])] \
    .pivot_table(index=["Outcome", "Threshold probability"], columns="Strategy", values="Net benefit").reset_index()
display(dca_sel)

with pd.ExcelWriter(os.path.join(OUT_DIR, "rules_and_dca.xlsx")) as w:
    rule_tab.to_excel(w, sheet_name="Rules_vs_model", index=False)
    dca_sel.to_excel(w, sheet_name="DCA_selected", index=False)
    dca_tab.to_excel(w, sheet_name="DCA_full", index=False)
print("Saved: rules_and_dca.xlsx, fig7_decision_curve.png/.pdf")